# 真实 IPYNB 的分块与 Embedding 实验

填写一个课程 `.ipynb` 路径后，本 Notebook 将完全通过项目工具执行：

`NotebookParser → DocumentIndexer → ParentChildChunker → DashScopeClient`

解析过程只读取 notebook 内容，不执行其中的代码 cell。图片输出会提取到 `data/assets/`。

In [1]:
from pathlib import Path
import math
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from app.clients.dashscope import DashScopeClient
from app.config import get_settings
from app.rag.chunker import ParentChildChunker
from app.rag.indexer import DocumentIndexer
from app.rag.parsers.notebook import NotebookParser

# 在这里填写待测试的 .ipynb 文件路径，可以使用绝对路径或项目根目录的相对路径。
NOTEBOOK_PATH = "/Users/Link/Course/CS5489 Machine learning/Lecture1/Lecture1a.ipynb"
QUERY = "这份课件主要讲了什么？"

settings = get_settings()
parser = NotebookParser.from_settings(settings)
chunker = ParentChildChunker.from_settings(settings)
client = DashScopeClient(settings)
indexer = DocumentIndexer(chunker, client)

In [2]:
if not NOTEBOOK_PATH.strip():
    raise ValueError("请先在 NOTEBOOK_PATH 中填写一个 .ipynb 文件路径")

source_path = Path(NOTEBOOK_PATH).expanduser()
if not source_path.is_absolute():
    source_path = PROJECT_ROOT / source_path

document = parser.parse(source_path)
print(f"文档标题: {document.title}")
print(f"文档 ID: {document.document_id}")
print(f"来源文件: {document.source_path}")
print(f"内容块数量: {len(document.blocks)}")
for block in document.blocks[:12]:
    preview = block.text.replace("\n", " ")[:100]
    print(
        f"{block.kind.value:>8} | cell={block.cell_id} | "
        f"section={' > '.join(block.section_path)} | {preview}"
    )

文档标题: CS 5489 Machine Learning
文档 ID: document_a8ab2817931fced4175e83b9
来源文件: /Users/Link/Course/CS5489 Machine learning/Lecture1/Lecture1a.ipynb
内容块数量: 273
markdown | cell=cell-0 | section=CS 5489 Machine Learning | # CS 5489 Machine Learning
markdown | cell=cell-0 | section=Lecture 1a: Python Tutorial | # Lecture 1a: Python Tutorial
markdown | cell=cell-1 | section=Why Python? | # Why Python? - General-purpose high-level programming language - Design philosophy emphasizes progr
markdown | cell=cell-2 | section=What is special about Python? | # What is special about Python? - Object-oriented: everything is an object - Clean: usually one way 
markdown | cell=cell-3 | section=Applications for Python | # Applications for Python - Scientific Computing   - numpy, scipy, ipython - Data Science, Deep Lear
markdown | cell=cell-4 | section=Disadvantages of Python | # Disadvantages of Python - Not as fast as Java or C - However, you can call C-compiled libraries fr
markdown | cell=cell-5 | sect

In [3]:
prepared = await indexer.prepare_document(document)
parents = prepared.parents
embedded_children = prepared.children

print(f"父块数量: {len(parents)}")
print(f"子块数量: {len(embedded_children)}")
print(f"Embedding 模型: {prepared.embedding_model}")
print(f"dense 维度: {prepared.embedding_dimension}")
print(f"文档向量 Token 用量: {prepared.total_tokens}")
for index, item in enumerate(embedded_children[:12]):
    child = item.chunk
    preview = child.text.replace("\n", " ")[:120]
    print(
        f"child[{index}] tokens={child.token_count}, "
        f"blocks={child.block_ids}, sparse={len(item.sparse)} | {preview}"
    )

父块数量: 7
子块数量: 18
Embedding 模型: qwen3.7-text-embedding
dense 维度: 1024
文档向量 Token 用量: 9578
child[0] tokens=551, blocks=['cell-0:markdown:0', 'cell-0:markdown:1', 'cell-1:markdown:0', 'cell-2:markdown:0', 'cell-3:markdown:0', 'cell-4:markdown:0', 'cell-5:markdown:0', 'cell-6:markdown:0'], sparse=488 | [课程位置] CS 5489 Machine Learning  [说明] # CS 5489 Machine Learning  [说明] # Lecture 1a: Python Tutorial  [说明] # Why Python?
child[1] tokens=451, blocks=['cell-6:markdown:0', 'cell-7:markdown:0', 'cell-8:markdown:0', 'cell-9:markdown:0', 'cell-10:markdown:0'], sparse=887 | [课程位置] CS 5489 Machine Learning  [说明] Running Python =============== - Interactive shell (ipython)   - good for learning
child[2] tokens=421, blocks=['cell-10:markdown:0', 'cell-11:markdown:0', 'cell-12:markdown:0'], sparse=470 | [课程位置] CS 5489 Machine Learning > Jupyter tips  [说明] # Jupyter tips - Keyboard shortcuts   - there are a lot of keyboard
child[3] tokens=584, blocks=['cell-13:markdown:0', 'cell-14:markdown:0', 'cell-

In [4]:
query_embedding = await client.embed_query(
    QUERY,
    instruct="Retrieve relevant passages from course materials.",
)

def cosine_similarity(left, right):
    numerator = sum(a * b for a, b in zip(left, right, strict=True))
    left_norm = math.sqrt(sum(value * value for value in left))
    right_norm = math.sqrt(sum(value * value for value in right))
    return numerator / (left_norm * right_norm)

ranking = sorted(
    (
        (cosine_similarity(query_embedding.dense, item.dense), index, item.chunk)
        for index, item in enumerate(embedded_children)
    ),
    reverse=True,
)

print(f"查询: {QUERY}")
for score, index, child in ranking[:10]:
    preview = child.text.replace("\n", " ")[:160]
    print(f"{score:.4f} | child[{index}] | {preview}...")

top_sparse = sorted(query_embedding.sparse, key=lambda item: item.value, reverse=True)[:12]
print("\n查询 sparse 高权重词项:")
print([(item.token, round(item.value, 4)) for item in top_sparse])

查询: 这份课件主要讲了什么？
0.4906 | child[17] | [课程位置] CS 5489 Machine Learning  [说明] # Exception Handling - Catching an exception   - `except` block catches exceptions   - `else` block executes if no excepti...
0.4876 | child[15] | [课程位置] CS 5489 Machine Learning  [说明] # Outline 1. Python Intro 2. Python Basics (identifiers, types, operators) 3. Control structures (conditional and loops) 4...
0.4586 | child[9] | [课程位置] CS 5489 Machine Learning  [运行输出] {1, 2}  [代码] ```python print (sA | sB)    # set union ```  [运行输出] {1, 2, 4, 5, 6, 7}  [代码] ```python print (sA & sB)    ...
0.4470 | child[8] | [课程位置] CS 5489 Machine Learning  [说明] # Operators - Arithmetic: `+`, `-`, `*`, `/`, `%`, `**` (exponent), `//` (floor division)  [代码] ```python print(6/4)     #...
0.4409 | child[16] | [课程位置] CS 5489 Machine Learning > Saving Objects with Pickle  [运行输出] <__main__.MyList object at 0x1070817d0> [0]  [说明] - cPickle is a faster version (1,000 time...
0.4379 | child[11] | [课程位置] CS 5489 Machine Learning  [说明] # 

## 观察重点

- Markdown、代码、输出、HTML 表格和图片是否被识别为正确的内容块。
- 子块是否保留正确的 `cell_id` 对应 `block_id`。
- 子块长度、短代码块和章节边界是否合理。
- 与 `QUERY` 相关的子块是否排在相似度前列。
- sparse 高权重词项是否包含问题中的专业词语。